## Who Owns America's Debt?

This analysis looks at how total foreign holdings of U.S. Treasury securities have changed over time and which countries hold the most American debt, using data from the U.S. Department of the Treasury's [Treasury International Capital (TIC) system](https://home.treasury.gov/data/treasury-international-capital-tic-system). 

The dataset, "Major Foreign Holders of Treasury Securities," reports monthly country-level holdings in billions of dollars from March 2000 to December 2025, based on U.S. custodian reports on the country of the foreign entity holding the securities. It does not include multilateral holders such as the IMF, World Bank, or ECB.

Country-level groupings in the source data were discontinued after Feb 2016, this was adressed during the data cleaning part of the analysis.  

### Importing Libraries and Setting Up



In [1]:
# importing libraries
import pandas as pd

In [2]:
# reading the csv

df = pd.read_csv('foreign_debt.csv', header=None, skiprows=5)
df.head()
                

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,NaN,Dec,Nov,Oct,Sep,Aug,Jul,Jun,May,Apr,Mar,Feb,Jan,NaN
1,Country,2025,2025,2025,2025,2025,2025,2025,2025,2025,2025,2025,2025,NaN
2,NaN,------,------,------,------,------,------,------,------,------,------,------,------,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Japan,1185.5,1202.7,1200,1189.3,1180.4,1151.8,1147.9,1135,1134.5,1130.8,1125.9,1079.3,NaN


### Data Cleaning

There are multiple tables in the saved csv sheet arranged by year, so need to combine all of these to a single table for analysis. 

In [3]:
# combining the tables together
# ChatGPT wrote this code


tables = []

i = 0
while i < len(df):

    row = df.iloc[i]

    if str(row[0]) == 'Country':

        # clean months
        raw_months = df.iloc[i-1].tolist()[1:]
        months = [str(m).strip() for m in raw_months if pd.notna(m)]

        # make months unique
        seen = {}
        unique_months = []
        for m in months:
            if m in seen:
                seen[m] += 1
                unique_months.append(f"{m}_{seen[m]}")
            else:
                seen[m] = 0
                unique_months.append(m)

        year = row[1]

        data_start = i + 2  # skip dashes row                                                                             
        while data_start < len(df) and df.iloc[data_start].isnull().all():
           data_start += 1  # skip blank rows dynamically
        data = []
        j = data_start

        while j < len(df) and not df.iloc[j].isnull().all():
            data.append(df.iloc[j].tolist())
            j += 1

        if len(data) == 0:
            i = j
            continue

        temp = pd.DataFrame(data)

        # match correct number of columns
        expected_cols = 1 + len(unique_months)
        temp = temp.iloc[:, :expected_cols]

        temp.columns = ['Country'] + unique_months
        temp['Year'] = int(year)

        tables.append(temp)

        i = j

    else:
        i += 1

combined = pd.concat(tables, ignore_index=True)

combined.to_csv('combined_debt.csv', index=False)

Table is cluttered and not ordered in a way I can analyze the monthly debt flows for each year so need to clean and reorganize the table first. It has duplicate columns for some months, and in some years, there is extra spacing for countries etc.

In [4]:
# remove junk duplicate columns like Jun_1, Mar_1

combined = combined.loc[:, ~combined.columns.str.contains('_')]

In [5]:
# define correct month order

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


In [6]:
# keep only valid months that exist

valid_months = [months for months in month_order if months in combined.columns]

In [7]:
# reorder columns properly

combined = combined[['Country', 'Year'] + valid_months]

Because I am compiling a number of tables into one table the individual grand totals, no longer makes sence and would introduce errors to the analysis, so removing that. Also, some of the countries are double counted because they were in different groups such as 'oil exporters' until February 2016. So removing some of these too. 

In [8]:
# remove Grand Total rows, aggregate group rows, and strip whitespace from country names

combined['Country'] = combined['Country'].str.strip()
combined = combined[combined['Country'] != 'Grand Total']

# remove aggregate/group rows that double-count individual countries
combined = combined[~combined['Country'].isin(['Oil Exporters 3/', 'Carib Bnkng Ctrs 4/'])]

In [9]:
# adding a new column for country totals

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# converting all month columns to numeric
combined[months] = combined[months].apply(pd.to_numeric, errors='coerce')

Because the United Kingdom included Channel island Isle of Man until February 2026, it was named with "/2". Cleaning this to make "United Kingdom" one continous line throughout. No separate Channel Islands or Isle of Man entries exist in the data so it is not possible to combine the data together throughout, which would have been more accurate. 


In [10]:
# combining the two UK catergories together

combined['Country'] = combined['Country'].replace('United Kingdom 2/', 'United Kingdom')   

Belgium-Luxembourg" was a single row in 2000–2001, then split into separate rows from 2002 onward with a "Belgium  5/" in 2002. This has to be made consistant for the analysis.


In [11]:
# combining different Belgium rows together, over time

combined['Country'] = combined['Country'].replace('Belgium  5/', 'Belgium')




Belgium and Luxembourg was grouped together until February 2016, and the row was named different names over time. Making it one name and then adding up the two countries until 2025, for consistancy. 

In [12]:
# sum Belgium and Luxembourg into one row from 2002

bel_lux = combined[combined['Country'].isin(['Belgium', 'Luxembourg'])] #filter only the Belgium and Luxemberg rows. 
bel_lux = bel_lux.groupby('Year')[months].sum()
bel_lux['Country'] = 'Belgium-Luxembourg' #adding the new row to the dataset
bel_lux = bel_lux.reset_index()

In [13]:
# removing the original Belgium and Luxembourg rows

combined = combined[~combined['Country'].isin(['Belgium', 'Luxembourg'])] 

# append the merged rows 
                                                                                                                                                                         
combined = pd.concat([combined, bel_lux], ignore_index=True)

Saving the cleaned dataset.

In [14]:
# use the cleaned data and save it

df = combined.copy()
df.to_csv('combined_debt.csv', index=False)
df.tail(25)

,Country,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
856,All Other,2000,NaN,NaN,161.3,153.8,151.3,152.3,154.8,158.4,158.3,158.3,156.5,157.8
857,Belgium-Luxembourg,2002,21.8,21.2,22.4,21.8,21.7,34.4,34.5,36.1,37.0,36.1,36.9,36.9
858,Belgium-Luxembourg,2003,39.7,38.4,35.8,36.2,35.7,37.2,38.7,41.0,40.4,40.3,41.0,39.8
859,Belgium-Luxembourg,2004,40.6,40.6,40.5,39.4,39.1,56.1,55.6,55.5,55.5,56.3,57.3,58.4
860,Belgium-Luxembourg,2005,58.7,59.9,58.1,61.5,60.8,54.2,52.5,54.0,53.3,52.1,52.2,52.6
861,Belgium-Luxembourg,2006,51.9,53.4,53.7,54.9,55.6,72.5,74.5,72.9,72.5,72.8,73.5,73.6
862,Belgium-Luxembourg,2007,71.7,72.7,74.1,74.7,74.3,71.1,73.0,71.7,72.9,77.9,82.1,82.2
863,Belgium-Luxembourg,2008,79.2,93.5,102.6,94.3,87.6,118.0,104.7,105.2,119.9,116.4,109.5,113.2
864,Belgium-Luxembourg,2009,102.6,106.6,121.4,113.2,111.9,110.8,98.6,100.7,104.7,96.4,97.6,105.7
865,Belgium-Luxembourg,2010,96.5,94.8,101.0,95.4,93.2,132.4,127.7,125.9,114.6,106.6,110.2,116.7


### Analysis

I want to understand the top US creditors in 2025, and how this trend has changed over time. For this, I am calculating the country averages each year and then looking at how the major creditors changed over time. 

In [15]:
# calculating the country averages

df['Country_average'] = df[months].mean(axis=1, skipna=True)

In [16]:
# total foreign holdings of the US from 2000 to 2025

foreign_holdings = df.groupby('Year')['Country_average'].sum()
foreign_holdings.round(1)


Year
2000     967.8
2001     935.5
2002    1041.1
2003    1282.2
2004    1618.9
2005    1796.8
2006    1876.2
2007    2007.8
2008    2388.9
2009    3066.7
2010    3722.1
2011    4273.7
2012    5336.2
2013    5667.2
2014    6009.0
2015    6135.2
2016    6169.4
2017    6165.1
2018    6224.2
2019    6685.4
2020    7048.0
2021    7401.9
2022    7414.7
2023    7534.6
2024    8364.1
2025    9094.6
Name: Country_average, dtype: float64

In [17]:
# Which country held the most amount of debt in 2025

top_debt_holders = df[df['Year']==2025][['Country','Country_average']]
top_debt_holders = top_debt_holders[top_debt_holders['Country'] != 'All Other']    #removing all other because it doesn't really add to the analysis since I am interested in Individual countries or country groups holding US foreign debt                                                                                                                                                                                  
top_debt_holders = top_debt_holders[top_debt_holders['Country_average'] > 300].sort_values('Country_average', ascending=False) #filtering for > 0.3T to understand the larger creditors
top_debt_holders.head(15).round(1)

,Country,Country_average
0,Japan,1155.3
880,Belgium-Luxembourg,847.7
1,United Kingdom,836.0
2,"China, Mainland",722.3
4,Cayman Islands,428.9
3,Canada,423.3
5,France,371.1
6,Ireland,334.2
7,Taiwan,308.7
8,Switzerland,301.9


Now, checking how the debt held by each of the top creditors changed over time.

In [18]:
# total debt held over time by the top debt holders

top_countries = ['Japan', 'China, Mainland', 'United Kingdom', 'Cayman Islands', 'Ireland','Belgium-Luxembourg']
top_creditors = df[df['Country'].isin(top_countries)].pivot_table(index='Year', columns='Country', values='Country_average').round(1)
top_creditors

Country,Belgium-Luxembourg,Cayman Islands,"China, Mainland",Ireland,Japan,United Kingdom
Year,,,,,,
2000,25.4,NaN,65.3,NaN,318.0,66.1
2001,22.8,NaN,71.2,NaN,306.2,48.1
2002,30.1,NaN,95.0,8.0,345.3,67.8
2003,38.7,NaN,142.0,12.2,457.9,76.9
2004,49.6,NaN,188.4,15.3,664.6,82.6
2005,55.8,NaN,272.8,19.5,675.1,104.7
2006,65.1,NaN,358.9,16.3,629.4,107.1
2007,74.9,NaN,445.9,14.6,607.3,125.0
2008,103.7,NaN,573.4,25.2,610.4,148.3


In [19]:
# total debt held by Japan 

Japan = df[df['Country']== 'Japan'][['Country_average','Year']].sort_values('Year')
Japan.round(1)

,Country_average,Year
839,318.0,2000
821,306.2,2001
797,345.3,2002
773,457.9,2003
749,664.6,2004
727,675.1,2005
703,629.4,2006
679,607.3,2007
652,610.4,2008
620,709.5,2009


In [20]:
# total debt held by China

China = df[df['Country']== 'China, Mainland'][['Country_average','Year']].sort_values('Year')
China.round(1)

,Country_average,Year
840,65.3,2000
822,71.2,2001
798,95.0,2002
774,142.0,2003
750,188.4,2004
728,272.8,2005
704,358.9,2006
680,445.9,2007
651,573.4,2008
619,859.1,2009


Since there is a clear trend of Japan overtaking China to be the top debtholder over time, this will be one chart.


In [ ]:
# taking both to a single table

china_vs_japan = df[df['Country'].isin(['Japan', 'China, Mainland'])].pivot_table(index='Year', columns='Country', values='Country_average').round(1)                                                                                                                           
china_vs_japan


Country,"China, Mainland",Japan
Year,,
2000,65.3,318.0
2001,71.2,306.2
2002,95.0,345.3
2003,142.0,457.9
2004,188.4,664.6
2005,272.8,675.1
2006,358.9,629.4
2007,445.9,607.3
2008,573.4,610.4


Sinece Belgium, Luxemberg, Cayman Islnds and Ireland are 'tax havens' I want to look at how they have performed as a group over time.

In [22]:
# Total debt held by global financial centers such as Belgium- Luxemberg, Cayman Islnds and Ireland

fc = ['Belgium-Luxembourg', 'Cayman Islands', 'Ireland']
financial_centers = df[df['Country'].isin(fc)].groupby('Year')['Country_average'].sum().round(1)
financial_centers.name = 'Financial Centers'
financial_centers

Year
2000      25.4
2001      22.8
2002      38.1
2003      50.9
2004      64.9
2005      75.3
2006      81.5
2007      89.4
2008     128.8
2009     152.5
2010     156.8
2011     239.1
2012     500.1
2013     605.7
2014     849.6
2015     823.7
2016     889.9
2017     871.7
2018     866.9
2019     939.8
2020    1017.5
2021    1098.7
2022    1170.4
2023    1236.9
2024    1424.3
2025    1610.8
Name: Financial Centers, dtype: float64

In [23]:
#Total debt held by the UK

UK = df[df['Country'] == 'United Kingdom'][['Country_average', 'Year']].sort_values('Year') 
UK.round(1)

,Country_average,Year
841,66.1,2000
825,48.1,2001
799,67.8,2002
775,76.9,2003
751,82.6,2004
729,104.7,2005
705,107.1,2006
681,125.0,2007
653,148.3,2008
621,130.1,2009


### Charts

Bringing all the Key players to one table to be exported for making the charts.

In [24]:
# all the key players in one Chart

key_destinations = ['Japan', 'China, Mainland', 'United Kingdom', 'Belgium-Luxembourg', 'Cayman Islands', 'Ireland']
all_debtors = df[df['Country'].isin(key_destinations)].pivot_table(index='Year', columns='Country', values='Country_average').round(1)
all_debtors['Financial Centers'] = all_debtors[['Belgium-Luxembourg', 'Cayman Islands', 'Ireland']].sum(axis=1).round(1) # regrouping the financial centers
all_debtors = all_debtors.drop(columns=['Cayman Islands', 'Ireland'])# dropping the individual rows for Cayman Islands and Ireland so that they are not counted twice
all_debtors['Total'] = all_debtors.index.map(foreign_holdings).round(1)

# convert from billions to trillions

numeric_cols = all_debtors.columns
all_debtors[numeric_cols] = (all_debtors[numeric_cols] / 1000).round(2)

#saving to csv

all_debtors.to_csv('chart6_all_lines.csv')
all_debtors

Country,Belgium-Luxembourg,"China, Mainland",Japan,United Kingdom,Financial Centers,Total
Year,,,,,,
2000,0.03,0.07,0.32,0.07,0.03,0.97
2001,0.02,0.07,0.31,0.05,0.02,0.94
2002,0.03,0.10,0.35,0.07,0.04,1.04
2003,0.04,0.14,0.46,0.08,0.05,1.28
2004,0.05,0.19,0.66,0.08,0.06,1.62
2005,0.06,0.27,0.68,0.10,0.08,1.80
2006,0.07,0.36,0.63,0.11,0.08,1.88
2007,0.07,0.45,0.61,0.12,0.09,2.01
2008,0.10,0.57,0.61,0.15,0.13,2.39
